# RNN-AGT: core model demonstration## What changed in this notebookThe model, loss, sampler, metrics and data generation used to be defined**inline in this notebook**, and the same block was copy-pasted into five othernotebooks. That is why two defects survived so long: fixing one copy left theothers untouched, and the copies had already drifted apart.All of that now lives in the `rnn_agt` package. This notebook only sets up anexperiment and reports it.Two fixes are inherited automatically:1. **Censoring now reaches the outcome.** The old `prepare_subjects_for_nn`   passed `subj['log_gaps']` — the *latent, uncensored* gap times — to the   model, while `delta` said some records were censored. Padding past the   censoring point was passed through as real data too.2. **The WRS normalization `1/(K_i* K_l*)` is applied.** The old loss was a   plain Gehan rank loss. The subject-level weight is the mechanism that   handles induced dependent censoring, so without it the estimating function   is biased toward subjects with many events.`simulation/defect_impact.ipynb` measures how much both defects changed the numbers.

In [ ]:
import os, syssys.path.insert(0, os.path.abspath(".."))   # repository root, so `rnn_agt` importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport rnn_agtfrom rnn_agt import data as Dfrom rnn_agt.seeds import make_seedsfrom rnn_agt.train import TrainConfig, train_model, predictfrom rnn_agt.metrics import evaluateprint("rnn_agt", rnn_agt.__version__)

## 1. Verify the installation before trusting any numberFive correctness checks. If any fails, stop and resolve it rather than working around it.

In [ ]:
from rnn_agt.diagnostics import (check_cindex_agreement, check_censoring_leak,                                 check_predictability, check_subsampling_unbiasedness,                                 summarise_checks)from rnn_agt.models import RNNAGTseeds = make_seeds(20260903)rng = seeds.data()raw = D.generate_subjects(80, D.f_interaction, "normal", rng, D.DEPENDENCE_SPECS["ar1"])raw = D.apply_censoring(raw, 50.0, rng)subs = D.to_model_subjects(raw)pred = np.random.default_rng(0).normal(size=(len(subs), max(len(s["log_gaps"]) for s in subs)))seeds.seed_torch()demo_model = RNNAGT(3, hidden_dim=16, gru_layers=1)print(summarise_checks({    "C-index matches reference loop": check_cindex_agreement(subs, pred),    "no latent-gap leak":             check_censoring_leak(raw),    "subsampling unbiased":           check_subsampling_unbiasedness(subs, 3, n_draws=400, s=4),    "predictor is (A3)-predictable":  check_predictability(demo_model, subs, 3),}))

## 2. Generate data`tau` is now *solved for* to hit a target censoring fraction. The old code hardcoded `tau=3000` and reported whatever fraction came out, which made the 25/50/65% columns of Tables 1-3 approximate.

In [ ]:
seeds = make_seeds(42)rng = seeds.data()tau = D.calibrate_tau(1000, D.f_interaction, "normal", rng,                      target_censoring=0.50,                      dependence=D.DEPENDENCE_SPECS["ar1"])print(f"tau for 50% censoring: {tau:.1f}")train_subjects = D.make_dataset(1000, "interaction", "normal", rng,                                dependence="ar1", tau=tau)test_subjects  = D.make_dataset(2000, "interaction", "normal", rng,                                dependence="ar1", tau=tau)for name, s in (("train", train_subjects), ("test", test_subjects)):    n_rec = sum(len(x["log_gaps"]) for x in s)    n_ev  = sum(int(x["delta"].sum()) for x in s)    print(f"{name}: {len(s)} subjects, {n_rec} records, "          f"{1 - n_ev / n_rec:.1%} censored")

## 3. Fit the three model classesThe same trainer drives all three, so a difference between rungs is attributable to the capability that rung adds and nothing else.Note `lr` differs for AFT-WRS: three parameters against the GRU's tens of thousands means it barely moves at `3e-4`. An undertrained comparator would flatter RNN-AGT for the wrong reason.

In [ ]:
configs = {    "AFT-WRS": TrainConfig(model="aft_wrs", lr=1e-2, epochs=10, pair_sample_s=30),    "NN-AFT":  TrainConfig(model="nn_aft",  lr=3e-4, epochs=10, pair_sample_s=30,                           hidden_dim=64, gru_layers=2),    "RNN-AGT": TrainConfig(model="rnn_agt", lr=3e-4, epochs=10, pair_sample_s=30,                           hidden_dim=64, gru_layers=2),}results = {}for name, cfg in configs.items():    res = train_model(train_subjects, test_subjects, 3, cfg, make_seeds(11),                      verbose=False)    results[name] = res    print(f"{name:8s} params={res.n_params:7,d}  "          f"test C={res.metrics['test_cindex']:.3f}  "          f"test AMSE={res.metrics['test_amse']:.2f}")

## 4. Training-set metricsComputed by a single forward pass in evaluation mode *after* fitting, never accumulated during optimization. The censoring distribution is re-estimated per partition, so train and test values are not two draws of the same quantity — their IPCW weights are on different scales. That is why a test C-index can exceed a training one without anything being wrong (Section 5.2).

In [ ]:
rows = []for name, res in results.items():    rows.append({        "model": name,        "train C": res.metrics["train_cindex"],        "test C":  res.metrics["test_cindex"],        "train AMSE": res.metrics["train_amse"],        "test AMSE":  res.metrics["test_amse"],        "params": res.n_params,    })pd.DataFrame(rows).round(3)